# DeepWater: Water Level Estimation from Imagery

This notebook provides a complete training pipeline for water level estimation using computer vision.

**Authors:** Keegan Johnson, Forrest Peterson, Elice Priyadarshini, Jeffrey Weisinger  
**Course:** CS771 - Machine Learning, UW-Madison

## Quick Start
1. Run the Setup cells to install dependencies
2. Upload your data or use the provided data collection pipeline
3. Train the model
4. Evaluate and visualize results

## 1. Setup and Installation

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# Install dependencies
!pip install -q timm albumentations dataretrieval httpx python-dateutil pytz tqdm

In [ ]:
# Clone the repository (if running on Colab)
import os

# Check if we're in Colab
IN_COLAB = 'google.colab' in str(get_ipython())

if IN_COLAB:
    # Mount Google Drive for persistent storage
    from google.colab import drive
    drive.mount('/content/drive')
    
    # Clone repo if not exists
    if not os.path.exists('deepwater'):
        # You would typically clone from your repo:
        # !git clone https://github.com/your-repo/deepwater.git
        print("Please upload the deepwater folder or clone from your repository")
    
    %cd deepwater

# Add to path
import sys
sys.path.insert(0, '.')

In [ ]:
# Import deepwater modules
from deepwater.data import (
    WaterLevelDataset, 
    split_dataset, 
    create_dataloaders,
    SiteDiscovery,
    DataCollector,
)
from deepwater.models import (
    create_model, 
    create_model_from_config,
    MODEL_CONFIGS,
)
from deepwater.training import (
    WaterLevelTrainer,
    create_optimizer,
    create_scheduler,
)
from deepwater.utils import (
    evaluate_model,
    print_metrics,
    plot_predictions,
    plot_training_history,
    count_parameters,
    format_parameters,
)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

print("Imports successful!")

## 2. Configuration

Adjust these settings based on your GPU and requirements.

In [ ]:
# Configuration
CONFIG = {
    # Data
    'data_csv': 'data/images_and_data.csv',  # Your data file
    'images_dir': 'data/images',              # Image directory
    'output_dir': 'outputs',                  # Output directory
    
    # Model
    'model_type': 'siamese',                  # 'siamese' or 'triplet'
    'backbone': 'vit_small_patch16_224',      # See MODEL_CONFIGS for options
    
    # Training
    'batch_size': 16,                         # Adjust based on GPU memory
    'num_epochs': 50,                         # Number of training epochs
    'learning_rate': 1e-4,
    'weight_decay': 0.01,
    
    # Hardware
    'device': 'cuda' if torch.cuda.is_available() else 'cpu',
    'num_workers': 2,                         # DataLoader workers
    'use_amp': True,                          # Mixed precision training
}

print(f"Using device: {CONFIG['device']}")
print(f"Available model configs: {list(MODEL_CONFIGS.keys())}")

## 3. Data Collection (Optional)

If you don't have data yet, use this section to discover sites and collect data.

In [ ]:
# Discover best camera sites
COLLECT_NEW_DATA = False  # Set to True to collect new data

if COLLECT_NEW_DATA:
    with SiteDiscovery(
        min_images=100,
        min_elevation_range_ft=1.0,
    ) as discovery:
        # Discover sites in the Upper Midwest
        sites = discovery.discover_sites(
            states=['WI', 'MN', 'IL', 'MI', 'IA'],
            max_sites=100,
            parallel=True,
        )
        
        # Filter to best sites
        best_sites = discovery.filter_sites(sites, min_quality_score=30)
        
        # Save results
        discovery.save_sites(best_sites[:30], 'data/best_sites.json')
        
        print(f"Found {len(best_sites)} good sites")
        
        # Show top 5
        df = discovery.to_dataframe(best_sites[:5])
        display(df[['camera_id', 'site_id', 'num_images', 'elevation_range_ft', 'quality_score']])

In [ ]:
# Collect data from selected sites
if COLLECT_NEW_DATA:
    import json
    
    with open('data/best_sites.json', 'r') as f:
        sites = json.load(f)
    
    # Collect from top 5 sites (for quick demo)
    sites_to_collect = [
        {'camera_id': s['camera_id'], 'site_id': s['site_id']}
        for s in sites[:5]
    ]
    
    with DataCollector('data/collected') as collector:
        collector.collect_multiple_sites(sites_to_collect, parallel=False)
        combined_df = collector.create_combined_dataset()
        
        print(f"Collected {len(combined_df)} total samples")
        
    # Update config to use new data
    CONFIG['data_csv'] = 'data/collected/combined_dataset.csv'
    CONFIG['images_dir'] = 'data/collected'

## 4. Load and Prepare Data

In [ ]:
# Check data
data_path = Path(CONFIG['data_csv'])
if not data_path.exists():
    print(f"Data file not found: {data_path}")
    print("Please upload your data or run the collection cell above.")
else:
    df = pd.read_csv(data_path)
    print(f"Loaded {len(df)} samples")
    print(f"\nColumns: {df.columns.tolist()}")
    
    # Find elevation column
    elev_col = '00065' if '00065' in df.columns else 'gage_height_ft'
    print(f"\nElevation range: {df[elev_col].min():.2f} - {df[elev_col].max():.2f} ft")
    print(f"Elevation std: {df[elev_col].std():.2f} ft")
    
    # Plot distribution
    fig, ax = plt.subplots(figsize=(10, 4))
    df[elev_col].hist(bins=50, ax=ax)
    ax.set_xlabel('Gage Height (ft)')
    ax.set_ylabel('Count')
    ax.set_title('Water Level Distribution')
    plt.show()

In [ ]:
# Split dataset
train_csv, val_csv, test_csv = split_dataset(
    CONFIG['data_csv'],
    'data/splits',
    train_ratio=0.7,
    val_ratio=0.15,
    test_ratio=0.15,
    stratify_by='00065',
)

print(f"Train: {train_csv}")
print(f"Val: {val_csv}")
print(f"Test: {test_csv}")

In [ ]:
# Create dataloaders
train_loader, val_loader, test_loader = create_dataloaders(
    train_csv=train_csv,
    val_csv=val_csv,
    test_csv=test_csv,
    images_dir=CONFIG['images_dir'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    mode=CONFIG['model_type'],
    image_size=224,
)

print(f"Train batches: {len(train_loader)}")
print(f"Val batches: {len(val_loader)}")
print(f"Test batches: {len(test_loader)}")

In [ ]:
# Visualize a sample batch
batch = next(iter(train_loader))

fig, axes = plt.subplots(2, 4, figsize=(16, 8))

for i in range(4):
    # Image 1
    img1 = batch['image1'][i].permute(1, 2, 0).numpy()
    img1 = (img1 * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]).clip(0, 1)
    axes[0, i].imshow(img1)
    axes[0, i].set_title(f"Ref: {train_loader.dataset.denormalize_elevation(batch['elevation1'][i].item()):.2f} ft")
    axes[0, i].axis('off')
    
    # Image 2
    img2 = batch['image2'][i].permute(1, 2, 0).numpy()
    img2 = (img2 * [0.229, 0.224, 0.225] + [0.485, 0.456, 0.406]).clip(0, 1)
    axes[1, i].imshow(img2)
    axes[1, i].set_title(f"Query: {train_loader.dataset.denormalize_elevation(batch['elevation2'][i].item()):.2f} ft")
    axes[1, i].axis('off')

plt.suptitle('Sample Training Pairs (Reference → Query)', fontsize=14)
plt.tight_layout()
plt.show()

## 5. Create Model

In [ ]:
# Create model
model = create_model(
    model_type=CONFIG['model_type'],
    backbone=CONFIG['backbone'],
    pretrained=True,
    feature_dim=384,  # Match backbone
    hidden_dim=192,
    dropout=0.1,
    fusion_type='concat',
)

# Model info
params = count_parameters(model)
print(f"Model: {CONFIG['model_type']} with {CONFIG['backbone']}")
print(f"Total parameters: {format_parameters(params['total'])}")
print(f"Trainable: {format_parameters(params['trainable'])}")
print(f"Frozen: {format_parameters(params['frozen'])}")

## 6. Training

In [ ]:
# Create optimizer and scheduler
optimizer = create_optimizer(
    model,
    optimizer_type='adamw',
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
)

scheduler = create_scheduler(
    optimizer,
    scheduler_type='cosine',
    num_epochs=CONFIG['num_epochs'],
    warmup_epochs=5,
)

In [ ]:
# Create trainer
from datetime import datetime

experiment_name = f"{CONFIG['model_type']}_{datetime.now().strftime('%Y%m%d_%H%M')}"

trainer = WaterLevelTrainer(
    model=model,
    train_loader=train_loader,
    val_loader=val_loader,
    optimizer=optimizer,
    scheduler=scheduler,
    device=CONFIG['device'],
    output_dir=CONFIG['output_dir'],
    experiment_name=experiment_name,
    use_amp=CONFIG['use_amp'],
    gradient_clip=1.0,
    save_interval=10,
    early_stopping_patience=15,
)

print(f"Experiment: {experiment_name}")
print(f"Output directory: {CONFIG['output_dir']}/{experiment_name}")

In [ ]:
# Train!
history = trainer.train(num_epochs=CONFIG['num_epochs'])

In [ ]:
# Plot training history
plot_training_history(
    history,
    title=f'Training History - {experiment_name}',
    save_path=f"{CONFIG['output_dir']}/{experiment_name}/training_history.png"
)
plt.show()

## 7. Evaluation

In [ ]:
# Load best model
best_model_path = f"{CONFIG['output_dir']}/{experiment_name}/best_model.pt"
checkpoint = torch.load(best_model_path, map_location=CONFIG['device'])
model.load_state_dict(checkpoint['model_state_dict'])

print(f"Loaded best model from epoch {checkpoint['epoch'] + 1}")
print(f"Best validation loss: {checkpoint['best_val_loss']:.4f}")

In [ ]:
# Evaluate on test set
predictions, targets, metrics = evaluate_model(
    model,
    test_loader,
    device=CONFIG['device'],
    denormalize_fn=train_loader.dataset.denormalize_elevation,
)

print_metrics(metrics, title='Test Set Results')

In [ ]:
# Plot predictions vs targets
fig, ax = plot_predictions(
    predictions, targets,
    title='Test Set: Predictions vs Actual Water Levels',
    save_path=f"{CONFIG['output_dir']}/{experiment_name}/test_predictions.png"
)
plt.show()

In [ ]:
# Error distribution
from deepwater.utils import plot_error_distribution

fig, ax = plot_error_distribution(
    predictions, targets,
    title='Prediction Error Distribution',
    save_path=f"{CONFIG['output_dir']}/{experiment_name}/error_distribution.png"
)
plt.show()

## 8. Save Model for Production

If the model performs well, save it for deployment.

In [ ]:
# Save final model with metadata
import json

output_path = f"{CONFIG['output_dir']}/{experiment_name}"

# Save test metrics
with open(f"{output_path}/test_results.json", 'w') as f:
    json.dump(metrics, f, indent=2)

# Save config
with open(f"{output_path}/config.json", 'w') as f:
    json.dump(CONFIG, f, indent=2)

# Save normalization stats for inference
norm_stats = {
    'elevation_mean': train_loader.dataset.elevation_mean,
    'elevation_std': train_loader.dataset.elevation_std,
    'elevation_min': train_loader.dataset.elevation_min,
    'elevation_max': train_loader.dataset.elevation_max,
}
with open(f"{output_path}/normalization_stats.json", 'w') as f:
    json.dump(norm_stats, f, indent=2)

print(f"\nModel and results saved to: {output_path}")
print(f"\nFiles:")
for f in Path(output_path).iterdir():
    print(f"  - {f.name}")

## 9. Copy to Google Drive (Colab)

If running on Colab, copy results to Google Drive for persistence.

In [ ]:
if IN_COLAB:
    import shutil
    
    drive_path = f"/content/drive/MyDrive/deepwater_results/{experiment_name}"
    shutil.copytree(output_path, drive_path, dirs_exist_ok=True)
    
    print(f"Results copied to Google Drive: {drive_path}")

---

## Summary

This notebook demonstrates the complete training pipeline for water level estimation:

1. **Site Discovery**: Finding USGS camera sites with good data quality
2. **Data Collection**: Downloading images and synchronized gauge data
3. **Model Training**: Using Siamese/Triplet ViT architectures
4. **Evaluation**: Computing metrics and visualizing results

### Next Steps

- Try the **triplet** model for potentially better performance
- Experiment with **larger backbones** (vit_base_patch16_224)
- Add **water segmentation masks** from SAM2 for improved accuracy
- Train on **more sites** for better generalization